# **Hasarlı Fotoğrafları Onarmak için Inpainting**

**Bu derste hasarlı eski bir fotoğrafı alıp inpaint() fonksiyonunu kullanarak eski haline getireceğiz**


In [ ]:

import cv2
import numpy as np
from matplotlib import pyplot as plt


def imshow(title = "Image", image = None, size = 10):
    # Güvenlik: image None ise anlamlı bir hata ver
    if image is None:
        raise ValueError(f"imshow: '{title}' için verilen görüntü None (dosya yüklenememiş olabilir).")
    # Tek kanallı (grayscale) görüntüleri doğru şekilde göster
    if len(image.shape) == 2:
        h, w = image.shape[0], image.shape[1]
        aspect_ratio = h / w
        plt.figure(figsize=(size * aspect_ratio, size))
        plt.imshow(image, cmap='gray')
        plt.title(title)
        plt.axis('off')
        plt.show()
        return
    # Renkli BGR görüntüleri RGB'ye çevirerek göster
    h, w = image.shape[0], image.shape[1]
    aspect_ratio = h / w
    plt.figure(figsize=(size * aspect_ratio, size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()


**Telea Algoritması**

2004 yılında Alexandru Telea tarafından geliştirildi.\
Temel amacı, bir görüntüdeki hasarlı/eksik bölgeleri, çevredeki sağlam piksellerden bilgi yayarak doğal şekilde doldurmaktır.\
Çalışma prensibi:\
- Hasarlı alanın sınırından başlanır → önce kenarlara en yakın pikseller doldurulur.
Öncelik sırası (Fast Marching Method) kullanılır:
- Kenara yakın bölgeler daha önce işlenir.Böylece dıştan içe doğru düzgün bir ilerleme olur.
Her yeni piksel değeri, çevresindeki sağlam piksellerin ağırlıklı ortalaması alınarak hesaplanır.
Ağırlıklar, pikselin mesafesine, yönüne ve gradyanına (kenar bilgisini korumak için) göre belirlenir.
Bu işlem maskenin tamamı dolana kadar devam eder.

**Navier–Stokes Algoritması** 

Görüntü onarımını akışkan dinamiklerine dayalı bir problem olarak ele alır. Yöntem, Navier–Stokes kısmi diferansiyel denklemlerini kullanarak hasarlı bölgelerin içine izotropik olmayan difüzyon uygular. Bu sayede, sınırdaki kenar yönleri ve yapısal gradyanlar doğal biçimde içe doğru uzatılır.

Görüntünün hasarlı bölgeleri, akışkanın (örneğin boya veya mürekkep) boşlukları doldurması gibi onarılıyor

Sonuç olarak algoritma, özellikle kenar sürekliliğini ve geometrik yapıları korumada etkilidir; fakat Telea yöntemine kıyasla daha yüksek hesaplama maliyetine sahiptir.

Telea vs. Navier-Stokes

- Telea: 
  - Daha hızlıdır.
  - Küçük/orta ölçekli hasarlarda çok başarılıdır.
  - Özellikle eski fotoğraf restorasyonlarında tercih edilir.
- Navier-Stokes:
  - Akışkan dinamiği benzeri bir PDE (kısmi diferansiyel denklem) yaklaşımı kullanır.
  - Kenar çizgilerini daha iyi koruyabilir ama daha yavaştır.
  - Karmaşık ve büyük hasarlarda kullanılabilir.


Talea için cv2.INPAINT_TELEA Navier-Stokes için cv2.INPAINT_NS

In [ ]:
# Hasarlı fotoğrafımızı yükleyelim
image = cv2.imread('../files/images/america33.jpg')
if image is None:
    raise ValueError('Görüntü ../files/images/america33.jpg yüklenemedi - dosya yolunu kontrol edin.')
imshow('Original Damaged Photo', image)

# Hasarlı alanları işaretlediğimiz fotoğrafı yükleyin (grayscale)
marked_damages = cv2.imread('../files/images/america34.jpg', 0)
if marked_damages is None:
    raise ValueError('Marked damages resmi ../files/images/america34.jpg yüklenemedi - dosya yolunu kontrol edin.')
imshow('Marked Damages', marked_damages)

# Beyaz olmayan tüm renkleri siyaha dönüştürerek işaretli görüntümüzden bir maske yapalım
# 
ret, thresh1 = cv2.threshold(marked_damages, 254, 255, cv2.THRESH_BINARY)
imshow('Threshold Binary', thresh1)
# 254’ten büyük pikseller (yani neredeyse beyaz) 255 (tam beyaz) oluyor.
# Diğerleri 0 (siyah) oluyor.


# Eşikleme biraz daralttığı için w işaretlerimizi genişletelim (kalınlaştıralım)
# 
kernel = np.ones((7,7), np.uint8)
mask = cv2.dilate(thresh1, kernel, iterations = 1) # 7x7 boyutlu bir kare çekirdek ile genişletme yapılıyor.
imshow('Dilated Mask', mask)
cv2.imwrite("../files/images/america33_mask.png", mask)

# inpaint fonksiyonu maskede beyaz olan alanları dolduruyor (onarım yapıyor).
# mask tek kanallı (0/255) olmalı; bunu doğrulayalım
if len(mask.shape) != 2:
    raise ValueError('Mask tek kanallı değil - mask oluşturma aşamasını kontrol edin.')
restored = cv2.inpaint(image, mask, 3, cv2.INPAINT_TELEA)
# inpaint fonksiyonu maskede beyaz olan alanları dolduruyor (onarım yapıyor).
# 3 → arama yarıçapı (çevredeki piksellerden doldurma yapılacak mesafe).
# Telea algoritması kullanılıyor

imshow('Restored', restored)